# Debuging purpose notebook for games using game engine directly

In [9]:
import os
import sys
import sqlite3
import random
import json
import polars as pl
import time

# Add the parent directory of 'engine' to sys.path if not already present
engine_path = os.path.abspath(os.path.join(os.getcwd(), '..'))
if engine_path not in sys.path:
	sys.path.append(engine_path)

from importlib import reload

# Always reload modules to reflect code changes during development
from engine import game_engine as ge 
reload(ge)
import models
reload(models)
import player_ai.playerai as pai
reload(pai)

from models import GameState, PlayerState

DB_PATH = 'games.db'

In [10]:
CARDS_DB = ge.get_cardpool()
CARDS_DB.head(3)

faction,mana,advancing,shield,condition,effect,effect_number,rare,condeff_value,card_id,prompt,negative_prompt,name
str,i64,i64,i64,str,str,i64,bool,i64,str,str,str,str
"""Dwarves""",2,2,3,"""no_condition""","""advancing""",1,false,0,"""Dwa23_79c784""","""A small dark skin fat female …","""""","""Hanna Zanna"""
"""Dwarves""",2,1,3,"""no_condition""","""advancing""",2,false,-1,"""Dwa23_5cd6c5""","""A small fat female dwarf wea…","""""","""Filda Rurik"""
"""Dwarves""",2,0,3,"""no_condition""","""advancing""",3,false,-2,"""Dwa23_5a81c9""","""A small old slim female dwarf…","""""","""Korla Hildir"""


# 1. Test with fake game

First player create the game

In [11]:
# p1 = PlayerState(name='Anne', deck=[f'c{i}' for i in range(1, 16)])   # old quick way to fake cards ids
# Randomly choose a faction and select 15 cards from that faction
faction = random.choice(CARDS_DB['faction'].unique().to_list())
filtered_cards = CARDS_DB.filter(pl.col('faction') == faction)
selected_card_ids = random.sample(filtered_cards['card_id'].to_list(), 15)
p1 = PlayerState(name='Anne', deck=selected_card_ids)
p1_ai = pai.PlayerAI(p1)

GAME_ID = ge.create_new_game(player=p1.model_dump())
print(f'game.id {GAME_ID}')

game.id 26_08_22_12_14_20_k0O7q


Whenever you want you can call get_game (with GAME_ID) to check the crurrent state of the game 

In [12]:
print(ge.get_game(game_id=GAME_ID))

{"id":"26_08_22_12_14_20_k0O7q","players":{"Anne":{"name":"Anne","hand":null,"mana":null,"mana_spend":0,"deck":["Dem10_7acd1f","Dem43_3c2c5e","Dem21_259291","Dem10_d6366b","Dem10_06f9b6","Dem43_7471da","Dem32_eaddea","Dem32_f96e7a","Dem21_33234e","Dem32_09f301","Dem43_505294","Dem32_37bf14","Dem21_8958c1","Dem43_8fcb40","Dem21_ffca7d"],"discard":null,"current_position":0,"message":null,"messages_history":[],"action_chain":[],"dwelling":null,"pendings":[]}},"turn_order":[],"turn":1,"state":"waiting for 2nd player","message":null,"earth":[],"winner":null,"first_player_passed":false,"second_player_passed":false,"temperature":null,"day_night":null}


P2 connects to the same game<br>
❗answers from the game engine for a Player do not include the information of the other player.<br>
While, answer from **get_game** function return the whole game state

In [13]:
# Randomly choose a faction and select 15 cards from that faction
faction = random.choice(CARDS_DB['faction'].unique().to_list())
filtered_cards = CARDS_DB.filter(pl.col('faction') == faction)
selected_card_ids = random.sample(filtered_cards['card_id'].to_list(), 15)
p2 = PlayerState(name='Boris', deck=selected_card_ids)
p2_ai = pai.PlayerAI(p2)

resp = ge.p2_connect_to_game(player=p2.model_dump(), game_id=GAME_ID)
print(f'p2 connect response: {resp}')
print(f'game: {ge.get_game(game_id=GAME_ID)}')

Planet initialized: temperature = 11, day
p2 connect response: {"id":"26_08_22_12_14_20_k0O7q","players":{"Anne":{"name":"Anne","hand":[],"mana":[],"mana_spend":0,"deck":[],"discard":[],"current_position":0,"message":null,"messages_history":[],"action_chain":[],"dwelling":null,"pendings":[]},"Boris":{"name":"Boris","hand":["Dwa45_b56ae0","Dwa56_7c777a","Dwa23_29aa9c","Dwa56_5b3016","Dwa56_ffeda3","Dwa56_47fc99"],"mana":[],"mana_spend":0,"deck":["Dwa45_d48874","Dwa56_5c0308","Dwa34_2358ad","Dwa55_570d8d","Dwa34_fbec65","Dwa56_fd46e6","Dwa56_1fae51","Dwa34_0311b5","Dwa45_d46306"],"discard":[],"current_position":0,"message":null,"messages_history":[],"action_chain":[],"dwelling":null,"pendings":[]}},"turn_order":["Anne","Boris"],"turn":1,"state":"waiting for both players to put 3 cards in hand","message":null,"earth":[["DE","Anne","Boris"],["DE"],["DE"],["DE"],["DE"],["DE"],["JU"],["JU"],["JU"],["JU"],["JU"],["JU"],["OC"],["OC"],["OC"],["OC"],["OC"],["OC"],["MO"],["MO"],["MO"],["MO"],["MO

At this point, both players are connected to the same game and they have (6) cards in hand 

In [14]:
resp_gs = GameState.model_validate(json.loads(ge.get_game(game_id=GAME_ID)))
print(f'resp_gs.state: {resp_gs.state}')

p1 = PlayerState.model_validate(resp_gs.players[p1.name]) # update local p1 variable with gameengine response
print(f'p1.hand: {p1.hand}')
p1_ai.update_player_state(p1)   # update p1_ai with new state

p2 = PlayerState.model_validate(resp_gs.players[p2.name]) # update local p2 variable with gameengine response
print(f'p2.hand: {p2.hand}')
p2_ai.update_player_state(p2)   # update p2_ai with new state

resp_gs.state: waiting for both players to put 3 cards in hand
p1.hand: ['Dem43_8fcb40', 'Dem32_37bf14', 'Dem32_f96e7a', 'Dem43_505294', 'Dem21_33234e', 'Dem21_259291']
p2.hand: ['Dwa45_b56ae0', 'Dwa56_7c777a', 'Dwa23_29aa9c', 'Dwa56_5b3016', 'Dwa56_ffeda3', 'Dwa56_47fc99']


As we can see the state mention that we are "waiting for both players to put (3) cards in hand"<br>
So we are going to put 3 cards into mana for each player

In [15]:
p1.message = {
    'cards': p1_ai.put_mana(),     # cards selected by user - LIST (if move mode, max 1 card, if defend mode, no max)
    'to': 'mana',                           # destination selected by user - STRING [stopover_x, mana, pending_zone, dwelling, discard_pile]
    'mode': '',                             # mode selected by user - STRING ['', move, defend, dwelling_activation, pending, pass]
    'pendings': []                          # cards in pendings zone that has to be added to a normal move card - LIST
}
ge.handle_websocket_message(game_id=GAME_ID, player=p1) # p1 send message

p2.message = {
    'cards': p2_ai.put_mana(),     # cards selected by user - LIST (if move mode, max 1 card, if defend mode, no max)
    'to': 'mana',                           # destination selected by user - STRING [stopover_x, mana, pending_zone, dwelling, discard_pile]
    'mode': '',                             # mode selected by user - STRING ['', move, defend, dwelling_activation, pending, pass]
    'pendings': []                          # cards in pendings zone that has to be added to a normal move card - LIST
}
ge.handle_websocket_message(game_id=GAME_ID, player=p2) # p2 send message

resp_gs = GameState.model_validate(json.loads(ge.get_game(game_id=GAME_ID)))
print(f'message from game: {resp_gs.state}')

# update players state in AIs
p1 = PlayerState.model_validate(resp_gs.players[p1.name])
p2 = PlayerState.model_validate(resp_gs.players[p2.name])
p1_ai.update_player_state(p1)
p2_ai.update_player_state(p2)

message from game: turn 1 - waiting for first player (Anne) to play


From now game is initialized and the messages will always have the same struture as shown below.<br>
Simulating a random simple game until game is over:

In [16]:
resp_gs = GameState.model_validate(json.loads(ge.get_game(game_id=GAME_ID)))

def update(p1, p2, p1_ai, p2_ai):
    resp_gs = GameState.model_validate(json.loads(ge.get_game(game_id=GAME_ID)))
    # update players state in AIs
    p1 = PlayerState.model_validate(resp_gs.players[p1.name])
    p2 = PlayerState.model_validate(resp_gs.players[p2.name])
    p1_ai.update_player_state(p1)
    p2_ai.update_player_state(p2)
    return resp_gs, p1, p2, p1_ai, p2_ai

while game_not_over := not ('game over' in resp_gs.state):
    resp_gs, p1, p2, p1_ai, p2_ai = update(p1, p2, p1_ai, p2_ai)
    
    print(f'Turn {resp_gs.turn} --------------')
    current_turn = resp_gs.turn
    print(f'\tgame state: {resp_gs.state}')
    
    # Put mana phase if needed
    if resp_gs.state == "waiting for both players to mana or pass":
        # both players will put mana or not based on put_mana function
        for p, p_ai in [(p1, p1_ai), (p2, p2_ai)]:
            p_message = p_ai.put_mana(num_cards=1, in_turn=True)  # put 1 card to mana
            if p_message['mode'] == 'pass':
                print(f"{p.name} passed the mana phase")
            else:
                print(f"{p.name} put {p_message['cards'][0]} to {p_message['to']}")
            p.message = p_message
            ge.handle_websocket_message(game_id=GAME_ID, player=p) # p send message
        resp_gs, p1, p2, p1_ai, p2_ai = update(p1, p2, p1_ai, p2_ai)
        print(f'\tgame state: {resp_gs.state}')

    while resp_gs.turn == current_turn:
        
        if p1.name in resp_gs.state:
            current_player = PlayerState.model_validate(resp_gs.players[p1.name])
            p = p1
            p_message = p1_ai.play_card()
        elif p2.name in resp_gs.state:
            current_player = PlayerState.model_validate(resp_gs.players[p2.name])
            p = p2
            p_message = p2_ai.play_card()
        print(f"\t\t{current_player.name}'s played {p_message['cards']} ({p_message['mode']}) [{current_player.hand}]")
        p.message = p_message
        ge.handle_websocket_message(game_id=GAME_ID, player=p) # p send message

        resp_gs, p1, p2, p1_ai, p2_ai = update(p1, p2, p1_ai, p2_ai)
        current_player = PlayerState.model_validate(resp_gs.players[p.name])
        if p_message['mode'] == 'pass':
            print(f"\t\t\tpassed - mana spent so far this turn: {current_player.mana_spend}/{len(current_player.mana)}")
        else:
            card_cost = CARDS_DB.filter(pl.col('card_id').is_in(p_message['cards']))['mana'].sum()
            print(f"\t\t\tplayed card cost {card_cost} mana - total spent this turn: {current_player.mana_spend}/{len(current_player.mana)}")
        print(f'\tgame state: {resp_gs.state}')

        if resp_gs.state == 'game over':
            print(f"Game over - winner: {resp_gs.winner}")
            game_not_over = False
            break

        # time.sleep(0.05)

Turn 1 --------------
	game state: turn 1 - waiting for first player (Anne) to play
		Anne's played ['Dem32_f96e7a'] (move) [['Dem32_f96e7a', 'Dem21_33234e', 'Dem21_259291']]
			played card cost 3 mana - total spent this turn: 3/3
	game state: turn 1 - waiting for second player (Boris) to play
		Boris's played ['Dwa23_29aa9c'] (move) [['Dwa45_b56ae0', 'Dwa23_29aa9c', 'Dwa56_ffeda3']]
			played card cost 2 mana - total spent this turn: 2/3
	game state: turn 1 - waiting for first player (Anne) to play
		Anne's played [] (pass) [['Dem21_33234e', 'Dem21_259291']]
			passed - mana spent so far this turn: 3/3
	game state: turn 1 - waiting for second player (Boris) to play
		Boris's played [] (pass) [['Dwa45_b56ae0', 'Dwa56_ffeda3']]
Processing trip chain...
	Processing action index: 0
		Anne processing cards: {'cards': ['Dem32_f96e7a'], 'to': 'stopover_x', 'mode': 'move', 'pendings': []}
			adv: 3, mana: 3
			condition face_point_left not implemented, assuming met
			applying effect: grappli

In [ ]:
card_id = 'Mum44_fa57c0'
card = CARDS_DB.filter(pl.col('card_id') == card_id)
if not card.is_empty():
    print(f'Card found: {card}\n card mana cost: {card["mana"].item()}')

In [ ]:
resp_gs.earth[0]
if 'JU' in resp_gs.earth[0]:
    print(f'yes')